# Real Estate Feasibility Engine – Python App
## Step 6: The input screens (Streamlit)

**What this notebook does**
1. Installs **Streamlit** inside Colab (nothing is installed on your laptop).
2. Upgrades `engine/inputs.py` so every error knows **which field / row** caused it – and re-runs the 141 tests to prove nothing broke.
3. Writes `app.py` – the app with **9 screens**: Project & land · Scenarios & zones · Products · Product timing · Cost heads · Calculators · Payment plans · Tax & benchmarks · Checks & summary.
4. Starts the app and gives you a **web link** to open it in a new browser tab.

**What you get in the app**
* Forms and editable tables instead of Excel cells – dropdowns, checkboxes, ➕ to add rows (products, milestones, cost heads).
* Errors appear **right under the field** or in the **Check column of the row** that caused them; the sidebar shows ⚠️ counts per screen.
* A live summary in the sidebar (GDV, EBITDA %, developer IRR, peak funding, payback) that updates after every change.
* 💾 Save / 📂 open your project as a file, and ↺ reset to the sample.

Run cells top to bottom with **Shift + Enter**. Steps 1–5 must have been run once.

### Cell 1 – Connect to Google Drive and open the project folder

In [ ]:
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/feasibility_app'
except ImportError:
    BASE = os.path.abspath('feasibility_app')
os.chdir(BASE)
assert os.path.exists('engine/cashflow.py') and os.path.exists('tests'), 'Please run Steps 1–5 first.'
print('Working in:', BASE)

### Cell 2 – Install Streamlit (inside Colab, takes ~30 seconds)

In [ ]:
!pip install -q streamlit
import streamlit
print('Streamlit', streamlit.__version__, 'ready')

### Cell 3 – Upgrade the input checks (`engine/inputs.py`)
Same checks and messages as before; each error now also carries the name of the field or row it belongs to (`check_inputs()`), so the app can show it in the right place.

In [ ]:
%%writefile engine/inputs.py
"""
engine/inputs.py  -  STEP 1: the input structure of the Real Estate Feasibility Engine.

Everything a user types on the Excel INPUTS sheet is described here as Python "dataclasses"
(simple containers with named fields). The rest of the app (timeline, curves, collections,
cash flow, metrics) will only ever read from a `Project` object built from these classes.

Units used everywhere:  areas in sq ft (land in sq yd), rates in Rs, amounts in Rs Crore,
timing as QUARTER NUMBERS counted from the project start (Q1 = start quarter).
"""
from __future__ import annotations

import json
from dataclasses import dataclass, field, asdict
from datetime import date
from typing import Optional

# --------------------------------------------------------------------------------------------
# 1. CONSTANTS AND ALLOWED VALUES (the Excel "LISTS" sheet)
# --------------------------------------------------------------------------------------------
CRORE = 10_000_000          # 1 Crore = 1,00,00,000
HORIZON_Q = 40              # model length: 40 quarters = 10 years
MAX_BOOKING_LAG = 12        # booking-linked milestones: 0..12 quarters after booking

SQFT_PER_SQYD = 9.0
SQM_PER_SQYD = 0.83612736
SQYD_PER_ACRE = 4840.0
SQYD_PER_BIGHA = 3025.0     # Rajasthan pucca bigha - change for other states

DEAL_TYPES = ["JV", "Outright"]
ENTITIES = ["Company", "LLP"]
PROJECT_TYPES = ["Villa", "Plotted", "Apartments", "Mixed Development"]
ZONES = ["Villa Development", "Apartment Development", "Plotted Development", "Other / Clubhouse"]
CATEGORIES = ["Villa", "Apartment", "Studio", "Plot", "Commercial", "Institutional", "EWS/LIG"]
BASIC_CURVES = ["Even", "Front-loaded", "Back-loaded", "S-curve"]
LAND_CURVES = ["Upfront"] + BASIC_CURVES
COST_CURVES = LAND_CURVES + ["Follow Construction", "Follow Sales", "Follow Collections",
                             "Follow Project Life", "At Project End"]
MANUAL_CURVES = LAND_CURVES                      # curves that need start / end quarters
SPARE_BASES = ["Rs/sq ft (total saleable area)", "% of total revenue", "% of construction cost",
               "Lump sum (Rs Cr)", "Rs/sq yd (land area)"]
EDC_BASES = ["Rs/sq ft (total saleable area)", "% of construction cost", "Lump sum (Rs Cr)"]
CONTINGENCY_BASES = ["Construction cost", "All costs excl. land heads"]
TRIGGERS = ["Booking + quarters", "Construction % complete"]
LAUNCH_MODES = ["Fixed quarter", "Linked to another product", "After own construction start"]
CONS_MODES = ["Fixed quarter", "After own launch", "Linked to another product"]
LINK_EVENTS = ["Launch", "Sales end", "Construction start", "Construction 50%", "Construction end"]
BAR_AUTHORITIES = ["RIICO", "JDA"]
BAR_LAND_USES = ["Industrial", "Residential", "Commercial"]
COMPOUNDING = {"Annually": 1, "Half-yearly": 2, "Quarterly": 4, "Monthly": 12}

# Project type -> development zones that are calculated (Mixed = all zones)
ZONES_FOR_TYPE = {
    "Villa": ["Villa Development", "Other / Clubhouse"],
    "Apartments": ["Apartment Development", "Other / Clubhouse"],
    "Plotted": ["Plotted Development", "Other / Clubhouse"],
    "Mixed Development": ZONES,
}

# Construction cost heads: one per product category (key, display name, category)
CONSTRUCTION_HEADS = [
    ("villa", "Villa Construction Cost", "Villa"),
    ("apt", "APT Construction Cost", "Apartment"),
    ("studio", "Studio Construction cost with furnishing", "Studio"),
    ("plot", "Plotted Construction Development Cost", "Plot"),
    ("inst", "Institutional Plotted Development Cost", "Institutional"),
    ("comm", "Commercial Construction Cost", "Commercial"),
    ("ews_dev", "EWS/LIG plots development cost", "EWS/LIG"),
]


# --------------------------------------------------------------------------------------------
# 2. THE BUILDING BLOCKS (one class per section of the INPUTS sheet)
# --------------------------------------------------------------------------------------------
@dataclass
class ProjectInfo:                                   # INPUTS Section 1
    name: str
    location: str
    project_type: str
    start_date: date                                 # must be a quarter-end date (= Q1)
    deal_type: str
    entity: str
    scenario: str
    lifecycle_override_years: Optional[float] = None # blank = use derived project span


@dataclass
class Land:                                          # INPUTS Section 2
    area_sqyd: float
    market_rate: float                               # Rs / sq yd
    dlc_rate_sqm: float                              # Rs / sq m (government guideline rate)
    lo_share: float                                  # JV: landowner revenue share
    stamp_duty_jv: float                             # JV: % of DLC value
    purchase_rate: float                             # Outright: Rs / sq yd
    stamp_duty_outright: float                       # Outright: % of max(DLC, price)
    payment_curve: str = "Upfront"                   # Outright land payment
    payment_start_q: int = 1
    payment_end_q: int = 1


@dataclass
class Scenario:                                      # INPUTS Section 3 (one row)
    name: str
    sales_period_mult: float                         # 1.5 = sales take 50% longer
    price_change: float                              # -0.075 = prices 7.5% lower
    cost_change: float                               # +0.075 = construction 7.5% dearer
    delay_q: int                                     # quarters added to fixed launch/start


@dataclass
class Zone:                                          # INPUTS Section 4 (one row)
    name: str
    share: float                                     # share of land area


@dataclass
class Timing:                                        # INPUTS Section 5B (one row)
    launch_mode: str = "Fixed quarter"
    launch_link_product: Optional[str] = None
    launch_link_event: Optional[str] = None
    launch_offset: int = 0
    cons_mode: str = "Fixed quarter"
    cons_link_product: Optional[str] = None
    cons_link_event: Optional[str] = None
    cons_offset: int = 0


@dataclass
class Product:                                       # INPUTS Section 5 (one row)
    name: str
    zone: str
    category: str
    include: bool                                    # "Include? (your choice)"
    area_sqft: float
    rate: float                                      # base sales rate, Rs / sq ft
    construction_rate: float                         # Rs / sq ft of saleable area
    launch_q: int                                    # used if launch is "Fixed quarter" (and as fallback)
    sales_period_q: int
    sales_curve: str
    clp_plan: str
    construction_start_q: int                        # used if construction is "Fixed quarter"
    construction_duration_q: int
    construction_curve: str
    timing: Timing = field(default_factory=Timing)


@dataclass
class CostHead:                                      # INPUTS Section 6 (one row)
    key: str                                         # internal id, e.g. "legal"
    name: str
    rate1: Optional[float] = None                    # "Rate / input 1"
    rate2: Optional[object] = None                   # "Input 2" (number or a basis name)
    applicable: bool = True
    curve: str = "Follow Construction"
    start_q: Optional[int] = None
    end_q: Optional[int] = None
    basis: Optional[str] = None                      # spare rows only


@dataclass
class BarCalc:                                       # INPUTS Section 7A
    authority: str
    land_use: str
    free_far: float
    loading_factor: float
    reserved_price_sqm: float
    gst_applicable: bool
    gst_rate: float
    pct_table: dict                                  # {land_use: {authority: pct or None}}


@dataclass
class AifCalc:                                       # INPUTS Section 7B
    rate: float
    tenure_years: float
    compounding: str


@dataclass
class InitialInvestment:                             # INPUTS Sections 7C & 7D
    advance_pct: float                               # JV advance to LO, % of market land value
    dd_amount: float                                 # Rs Cr, on due diligence
    da_amount: float                                 # Rs Cr, at DA (token = balance)
    legal_share: float                               # share of legal paid before DA
    bar_share: float                                 # share of BAR paid upfront
    dev_cost_pre_da: float                           # Rs Cr
    marketing_pre_launch: float                      # Rs Cr
    payout_q: dict                                   # item -> quarter number
    outright_construction_share: float               # Outright: % of built-up construction
    outright_legal_share: float                      # Outright: % of legal


@dataclass
class Milestone:                                     # INPUTS Section 8 (one row of a plan)
    name: str
    trigger: str                                     # "Booking + quarters" | "Construction % complete"
    value: float                                     # quarters after booking OR construction fraction
    pct: float                                       # share of sale value


@dataclass
class ClpPlan:
    name: str
    milestones: list


@dataclass
class Finance:                                       # INPUTS Section 9
    tax_company: float
    tax_llp: float
    cost_of_equity: float
    benchmarks: dict


@dataclass
class Project:                                       # the whole INPUTS sheet
    info: ProjectInfo
    land: Land
    scenarios: list
    zones: list
    products: list
    costs: list
    bar: BarCalc
    aif: AifCalc
    initial: InitialInvestment
    catch_up_lag: int
    plans: list
    finance: Finance

    # ---------- small helpers used everywhere ----------
    def active_products(self) -> list:
        """Products that are included AND belong to a zone allowed by the project type."""
        allowed = ZONES_FOR_TYPE.get(self.info.project_type, [])
        return [p for p in self.products if p.include and p.zone in allowed]

    def scenario(self) -> Scenario:
        return next(s for s in self.scenarios if s.name == self.info.scenario)

    def plan(self, name: str) -> ClpPlan:
        return next(p for p in self.plans if p.name == name)

    def cost(self, key: str) -> CostHead:
        return next(c for c in self.costs if c.key == key)


# --------------------------------------------------------------------------------------------
# 3. SAVE / LOAD (a project is stored as a readable JSON text file)
# --------------------------------------------------------------------------------------------
def to_dict(project: Project) -> dict:
    d = asdict(project)
    d["info"]["start_date"] = project.info.start_date.isoformat()
    return d


def from_dict(d: dict) -> Project:
    info = dict(d["info"]); info["start_date"] = date.fromisoformat(info["start_date"])
    products = []
    for p in d["products"]:
        p = dict(p); p["timing"] = Timing(**p["timing"]); products.append(Product(**p))
    plans = [ClpPlan(pl["name"], [Milestone(**m) for m in pl["milestones"]]) for pl in d["plans"]]
    return Project(
        info=ProjectInfo(**info), land=Land(**d["land"]),
        scenarios=[Scenario(**s) for s in d["scenarios"]], zones=[Zone(**z) for z in d["zones"]],
        products=products, costs=[CostHead(**c) for c in d["costs"]],
        bar=BarCalc(**d["bar"]), aif=AifCalc(**d["aif"]), initial=InitialInvestment(**d["initial"]),
        catch_up_lag=d["catch_up_lag"], plans=plans, finance=Finance(**d["finance"]))


def save_project(project: Project, path: str) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(to_dict(project), f, indent=2, ensure_ascii=False)


def load_project(path: str) -> Project:
    with open(path, encoding="utf-8") as f:
        return from_dict(json.load(f))


# --------------------------------------------------------------------------------------------
# 4. INPUT CHECKS (the Excel "Model checks" for inputs) - returns a list of plain-English errors
# --------------------------------------------------------------------------------------------
def _is_int(x) -> bool:
    return isinstance(x, (int, float)) and float(x).is_integer()


@dataclass
class Issue:
    section: str      # INPUTS section, e.g. "Products"
    item: str         # the field / row it belongs to, e.g. a product name or "start_date"
    message: str


def validate(project: Project) -> list:
    """Plain-English error list, e.g. ['[CLP] Plan 'X' totals 95.00% - it must be exactly 100%.']"""
    return [f"[{i.section}] {i.message}" for i in check_inputs(project)]


def check_inputs(project: Project) -> list:
    """Same checks as validate(), but each problem also says WHICH field/row caused it (used by the app
    to show the error right next to that field)."""
    errors = []
    def err(section, msg, item=""): errors.append(Issue(section, item, msg))
    i = project.info

    # Section 1 - project
    d = i.start_date
    if not (d.month in (3, 6, 9, 12) and (date(d.year + (d.month == 12), d.month % 12 + 1, 1) - d).days == 1):
        err("Project", f"Start date {d} is not a quarter-end date (31-Mar, 30-Jun, 30-Sep, 31-Dec).", "start_date")
    for val, allowed, label in [(i.project_type, PROJECT_TYPES, "Project type"), (i.deal_type, DEAL_TYPES, "Deal type"),
                                (i.entity, ENTITIES, "Taxable entity")]:
        if val not in allowed: err("Project", f"{label} '{val}' must be one of {allowed}.", label)
    if i.scenario not in [s.name for s in project.scenarios]:
        err("Project", f"Active scenario '{i.scenario}' is not in the scenario table.", "scenario")
    if i.lifecycle_override_years is not None and i.lifecycle_override_years <= 0:
        err("Project", "Lifecycle override must be blank or a positive number of years.", "lifecycle")

    # Section 2 - land
    L = project.land
    if L.area_sqyd <= 0 or L.market_rate <= 0 or L.dlc_rate_sqm <= 0:
        err("Land", "Land area, market rate and DLC rate must be greater than zero.", "land_rates")
    if i.deal_type == "JV" and not (0 <= L.lo_share < 1):
        err("Land", "Landowner share must be between 0% and 99.9%.", "lo_share")
    if i.deal_type == "Outright":
        if L.purchase_rate <= 0: err("Land", "Purchase rate must be greater than zero.", "purchase_rate")
        if L.payment_curve not in LAND_CURVES: err("Land", f"Land payment curve must be one of {LAND_CURVES}.", "payment")
        end = L.payment_start_q if L.payment_curve == "Upfront" else L.payment_end_q
        if not (_is_int(L.payment_start_q) and 1 <= L.payment_start_q <= end <= HORIZON_Q):
            err("Land", "Land payment start/end quarters must be whole numbers with 1 <= start <= end <= 40.", "payment")

    # Section 3 - scenarios
    names = [s.name for s in project.scenarios]
    if len(names) != len(set(names)): err("Scenarios", "Scenario names must be unique.", "names")
    for s in project.scenarios:
        if s.sales_period_mult <= 0 or s.price_change <= -1 or s.cost_change <= -1 or not (_is_int(s.delay_q) and s.delay_q >= 0):
            err("Scenarios", f"Scenario '{s.name}' has an invalid value.", s.name)

    # Section 4 - zones
    if sum(z.share for z in project.zones) > 1 + 1e-9 or any(z.share < 0 for z in project.zones):
        err("Zones", "Zone shares must be >= 0 and total at most 100%.", "zones")

    # Section 5 - products
    pnames = [p.name for p in project.products]
    if len(pnames) != len(set(pnames)): err("Products", "Every product needs a different name (needed for links).", "names")
    active = project.active_products()
    if not active: err("Products", "No product is active for this project type - include a product whose zone matches.", "active")
    plan_names = [pl.name for pl in project.plans]
    for idx, p in enumerate(project.products):
        if not p.include: continue
        tag = f"Product '{p.name}'"
        if p.zone not in ZONES: err("Products", f"{tag}: zone must be one of {ZONES}.", p.name)
        if p.category not in CATEGORIES: err("Products", f"{tag}: category must be one of {CATEGORIES}.", p.name)
        if p.area_sqft <= 0 or p.rate < 0 or p.construction_rate < 0:
            err("Products", f"{tag}: area must be > 0 and rates >= 0.", p.name)
        for val, label in [(p.sales_period_q, "sales period"), (p.construction_duration_q, "construction duration")]:
            if not (_is_int(val) and val >= 1): err("Products", f"{tag}: {label} must be a whole number >= 1.", p.name)
        if p.sales_curve not in BASIC_CURVES or p.construction_curve not in BASIC_CURVES:
            err("Products", f"{tag}: sales/construction curve must be one of {BASIC_CURVES}.", p.name)
        if p.clp_plan not in plan_names: err("Products", f"{tag}: payment plan '{p.clp_plan}' does not exist.", p.name)
        t = p.timing
        if t.launch_mode not in LAUNCH_MODES or t.cons_mode not in CONS_MODES:
            err("Timing", f"{tag}: invalid launch / construction timing mode.", p.name)
        if t.launch_mode == "After own construction start" and t.cons_mode == "After own launch":
            err("Timing", f"{tag}: launch and construction cannot both depend on each other (circular).", p.name)
        above = pnames[:idx]
        for mode, prod, ev, label in [(t.launch_mode, t.launch_link_product, t.launch_link_event, "launch"),
                                      (t.cons_mode, t.cons_link_product, t.cons_link_event, "construction")]:
            if mode == "Linked to another product":
                if prod not in above: err("Timing", f"{tag}: {label} is linked to '{prod}', which must be a product listed ABOVE it.", p.name)
                if ev not in LINK_EVENTS: err("Timing", f"{tag}: {label} link event must be one of {LINK_EVENTS}.", p.name)
        for val, label in [(p.launch_q, "fixed launch quarter"), (p.construction_start_q, "fixed construction start quarter")]:
            if not (_is_int(val) and val >= 1): err("Products", f"{tag}: {label} must be a whole number >= 1 (also used as fallback).", p.name)

    # Section 6 - cost heads
    keys = [c.key for c in project.costs]
    for c in project.costs:
        if c.curve not in COST_CURVES: err("Costs", f"'{c.name}': phasing curve must be one of {COST_CURVES}.", c.key); continue
        if c.curve in MANUAL_CURVES and c.applicable:
            end = c.start_q if c.curve == "Upfront" else c.end_q
            if not (_is_int(c.start_q) and _is_int(end) and 1 <= c.start_q <= end <= HORIZON_Q):
                err("Costs", f"'{c.name}': start/end quarters must be whole numbers with 1 <= start <= end <= 40.", c.key)
        if c.rate1 is not None and isinstance(c.rate1, (int, float)) and c.rate1 < 0:
            err("Costs", f"'{c.name}': rate cannot be negative.", c.key)
        if c.key.startswith("spare") and c.applicable and c.basis not in SPARE_BASES:
            err("Costs", f"'{c.name}': basis must be one of {SPARE_BASES}.", c.key)
    if "cont" in keys and project.cost("cont").rate2 not in CONTINGENCY_BASES:
        err("Costs", f"Contingency base must be one of {CONTINGENCY_BASES}.", "cont")
    if "edc" in keys and project.cost("edc").rate2 not in EDC_BASES:
        err("Costs", f"External development basis must be one of {EDC_BASES}.", "edc")

    # Section 7 - calculators
    b = project.bar
    if b.authority not in BAR_AUTHORITIES or b.land_use not in BAR_LAND_USES:
        err("BAR", "Authority / land use not recognised.", "bar")
    elif not b.pct_table.get(b.land_use, {}).get(b.authority):
        err("BAR", f"No BAR % defined for {b.authority} / {b.land_use} - combination not allowed.", "bar")
    if b.loading_factor <= 0: err("BAR", "Loading factor must be greater than zero.", "bar")
    if project.aif.compounding not in COMPOUNDING or project.aif.rate < 0 or project.aif.tenure_years < 0:
        err("AIF", "AIF rate / tenure / compounding invalid.", "aif")
    ii = project.initial
    advance = ii.advance_pct * L.area_sqyd * L.market_rate / CRORE
    if ii.dd_amount + ii.da_amount > advance + 1e-9:
        err("Initial investment", f"DD + DA instalments ({ii.dd_amount + ii.da_amount:.2f} Cr) exceed the total advance ({advance:.2f} Cr).", "dd_da")
    if any(not (_is_int(q) and 1 <= q <= HORIZON_Q) for q in ii.payout_q.values()):
        err("Initial investment", "Payout quarters must be whole numbers between 1 and 40.", "payout_q")

    # Section 8 - payment plans
    if not (_is_int(project.catch_up_lag) and 0 <= project.catch_up_lag <= 4):
        err("CLP", "Catch-up lag must be a whole number from 0 to 4.", "catch_up_lag")
    if len(plan_names) != len(set(plan_names)): err("CLP", "Payment plan names must be unique.", "plan_names")
    for pl in project.plans:
        total = sum(m.pct for m in pl.milestones)
        if abs(total - 1) > 1e-6: err("CLP", f"Plan '{pl.name}' totals {total:.2%} - it must be exactly 100%.", pl.name)
        for m in pl.milestones:
            if m.trigger == "Booking + quarters":
                if not (_is_int(m.value) and 0 <= m.value <= MAX_BOOKING_LAG):
                    err("CLP", f"Plan '{pl.name}', '{m.name}': booking lag must be a whole number 0-12.", pl.name)
            elif m.trigger == "Construction % complete":
                if not (0 <= m.value <= 1): err("CLP", f"Plan '{pl.name}', '{m.name}': construction trigger must be 0%-100%.", pl.name)
            else:
                err("CLP", f"Plan '{pl.name}', '{m.name}': trigger must be one of {TRIGGERS}.", pl.name)
            if m.pct < 0: err("CLP", f"Plan '{pl.name}', '{m.name}': % cannot be negative.", pl.name)

    # Section 9 - finance
    f = project.finance
    if not (0 <= f.tax_company < 1 and 0 <= f.tax_llp < 1 and f.cost_of_equity > 0):
        err("Finance", "Tax rates must be 0%-99.9% and cost of equity > 0.", "finance")
    return errors


# --------------------------------------------------------------------------------------------
# 5. THE SAMPLE PROJECT (identical to the Excel template "Project Aurora")
# --------------------------------------------------------------------------------------------
def sample_project() -> Project:
    T = Timing
    products = [
        Product("Villas", "Villa Development", "Villa", True, 250_000, 11_000, 2_900, 3, 8, "Front-loaded", "CLP – Villa", 4, 12, "S-curve",
                T("Fixed quarter", cons_mode="After own launch", cons_offset=1)),
        Product("Apartments – Towers A & B", "Apartment Development", "Apartment", True, 600_000, 7_500, 2_900, 4, 12, "Even", "CLP – Apartment", 5, 16, "S-curve",
                T("Linked to another product", "Villas", "Launch", 1, "After own launch", cons_offset=1)),
        Product("Studios", "Apartment Development", "Studio", True, 40_000, 9_000, 3_800, 6, 8, "Even", "CLP – Apartment", 6, 12, "S-curve",
                T("Linked to another product", "Apartments – Towers A & B", "Construction start", 1, "After own launch", cons_offset=0)),
        Product("High-street retail", "Apartment Development", "Commercial", True, 30_000, 14_000, 2_900, 8, 8, "Back-loaded", "Down Payment Plan", 6, 12, "S-curve",
                T("Linked to another product", "Apartments – Towers A & B", "Launch", 4,
                  "Linked to another product", "Apartments – Towers A & B", "Construction start", 1)),
        Product("Residential plots", "Plotted Development", "Plot", False, 150_000, 4_500, 450, 10, 8, "Even", "Plot – Development Linked", 9, 8, "Even"),
        Product("Institutional plots", "Plotted Development", "Institutional", False, 30_000, 4_000, 400, 12, 8, "Even", "Plot – Development Linked", 9, 8, "Even"),
        Product("EWS / LIG plots", "Plotted Development", "EWS/LIG", False, 20_000, 1_500, 400, 10, 8, "Even", "Plot – Development Linked", 9, 8, "Even"),
    ]
    C = CostHead
    costs = [
        C("land", "Land Cost", curve="Follow Collections"),
        C("sd", "Stamp Duty", curve="Upfront", start_q=1, end_q=1),
        C("brok", "Brokerage", rate1=0.0118, curve="Upfront", start_q=1, end_q=1),
    ] + [C(k, n, curve="Follow Construction") for k, n, _ in CONSTRUCTION_HEADS] + [
        C("ovh", "Corporate Overheads (Rs/Sq Ft SA)", rate1=0.04, curve="Follow Project Life"),
        C("mkt", "Marketing & Sales @ %", rate1=0.07, rate2=0.05, curve="Follow Sales"),
        C("legal", "Legal & Compliance (Rs/Sq Ft SA)", rate1=75, curve="Front-loaded", start_q=1, end_q=8),
        C("ews", "EWS / LIG @ Rs/Sq Ft (Builtup)", rate1=400, rate2=0.05, curve="Follow Construction"),
        C("cont", "Contingency @ %", rate1=0.0, rate2="Construction cost", curve="Follow Construction"),
        C("edc", "External Development & Landscaping", rate1=0.0, rate2="Rs/sq ft (total saleable area)", curve="Follow Construction"),
        C("hotel", "Hotel Brand Key fee Rs 50K per Key for smaller brand", rate1=50_000, rate2=0, curve="Follow Construction"),
        C("furn", "Furnishing Cost for Studios", rate1=500, applicable=False, curve="Follow Construction"),
        C("gst", "GST cost in case of commercial/Plotting development", rate1=0.18, rate2=0.20, applicable=False, curve="At Project End"),
        C("aif", "AIF Cost", applicable=False, curve="Follow Project Life"),
        C("bar", "BAR / Premium FAR Cost", curve="Upfront", start_q=2, end_q=2),
    ]
    B, M = "Booking + quarters", "Construction % complete"
    plans = [
        ClpPlan("CLP – Villa", [Milestone("Earnest money", B, 0, .10), Milestone("Agreement to sell (ATS)", B, 1, .10),
            Milestone("Commencement of construction", M, .05, .10), Milestone("Ground plinth", M, .15, .10),
            Milestone("First floor roof casting", M, .35, .15), Milestone("Second floor roof casting", M, .55, .15),
            Milestone("Third floor roof casting", M, .75, .15), Milestone("Completion certificate (CC)", M, .95, .05),
            Milestone("Possession", M, 1.0, .10)]),
        ClpPlan("CLP – Apartment", [Milestone("Booking amount", B, 0, .10), Milestone("Agreement to sell (ATS)", B, 1, .10),
            Milestone("Foundation completion", M, .10, .10), Milestone("Podium / plinth", M, .20, .10),
            Milestone("Slab casting – 25% floors", M, .35, .10), Milestone("Slab casting – 50% floors", M, .50, .10),
            Milestone("Slab casting – 75% floors", M, .65, .10), Milestone("Top slab casting", M, .80, .10),
            Milestone("Brickwork & plaster", M, .90, .05), Milestone("Flooring & finishing", M, .95, .05),
            Milestone("Possession", M, 1.0, .10)]),
        ClpPlan("Plot – Development Linked", [Milestone("Booking amount", B, 0, .10), Milestone("Agreement to sell (ATS)", B, 1, .20),
            Milestone("Roads & services start", M, .25, .20), Milestone("Services 60% complete", M, .60, .20),
            Milestone("Possession / registry", M, 1.0, .30)]),
        ClpPlan("Down Payment Plan", [Milestone("Booking amount", B, 0, .10), Milestone("Down payment", B, 1, .80),
            Milestone("Possession", M, 1.0, .10)]),
        ClpPlan("CLP – 17-Floor Tower", [Milestone("Booking amount", B, 0, .10), Milestone("Agreement to sell (ATS)", B, 1, .10),
            Milestone("Excavation", M, .03, .05), Milestone("Foundation / raft", M, .08, .05)]
            + [Milestone(f"Slab casting – floor {k}", M, round(0.12 + 0.03625 * (k - 1), 5), .025) for k in range(1, 18)]
            + [Milestone("Brickwork", M, .78, .05), Milestone("Plaster & MEP", M, .85, .05),
               Milestone("Flooring & finishing", M, .92, .075), Milestone("Possession", M, 1.0, .10)]),
    ]
    return Project(
        info=ProjectInfo("Project Aurora (Sample)", "Sample City, Rajasthan", "Mixed Development", date(2027, 6, 30), "JV", "Company", "Base Case"),
        land=Land(area_sqyd=30_000, market_rate=60_000, dlc_rate_sqm=13_000, lo_share=0.32, stamp_duty_jv=0.0275,
                  purchase_rate=60_000, stamp_duty_outright=0.088),
        scenarios=[Scenario("Best Case", 0.75, 0.05, 0.0, 0), Scenario("Base Case", 1.0, 0.0, 0.0, 0), Scenario("Worst Case", 1.5, -0.075, 0.075, 2)],
        zones=[Zone("Villa Development", 0.35), Zone("Apartment Development", 0.50), Zone("Plotted Development", 0.15), Zone("Other / Clubhouse", 0.0)],
        products=products, costs=costs,
        bar=BarCalc("JDA", "Residential", 2.0, 1.35, 12_000, False, 0.18,
                    {"Industrial": {"RIICO": 0.20, "JDA": None}, "Residential": {"RIICO": 0.30, "JDA": 0.20},
                     "Commercial": {"RIICO": 0.40, "JDA": 0.40}}),
        aif=AifCalc(0.18, 3, "Annually"),
        initial=InitialInvestment(0.10, 0.0, 0.0, 1.0, 0.25, 0.0, 0.0,
                                  {"token": 1, "dd": 1, "da": 2, "legal": 2, "dev": 2, "marketing": 2, "sd": 2, "brok": 2, "bar": 2},
                                  0.10, 0.10),
        catch_up_lag=1, plans=plans,
        finance=Finance(0.26, 0.35, 0.18, {"project_irr": 0.22, "developer_irr": 0.18, "equity_multiple": 2.0, "roce": 0.30,
                                           "cagr": 0.30, "pat_margin": 0.15, "ebitda_margin": 0.20}),
    )


### Cell 4 – Safety net: re-run all 141 tests after the change

In [ ]:
import subprocess, sys
res = subprocess.run([sys.executable, '-m', 'pytest', 'tests', '-q', '--tb=short'], capture_output=True, text=True)
print(res.stdout.strip().splitlines()[-1])
print('✅ Engine unchanged – safe to continue' if res.returncode == 0 else '❌ Something broke – details:\n' + res.stdout[-4000:])

### Cell 5 – Write the app (`app.py`)
One file, organised by screen. Every screen edits the same project in memory; after each change the whole engine re-runs
(a fraction of a second) and the screens redraw with fresh numbers and checks.

In [ ]:
%%writefile app.py
"""
app.py  -  STEP 6: the input screens of the Real Estate Feasibility Engine (Streamlit).

Run with:   streamlit run app.py
Every screen edits ONE Project object kept in memory (st.session_state). After every change the whole
engine re-runs (a fraction of a second), and any problem is shown right next to the field / row that caused it.
"""
from __future__ import annotations

import copy
import json
from datetime import date

import pandas as pd
import streamlit as st

from engine.inputs import (
    BAR_AUTHORITIES, BAR_LAND_USES, BASIC_CURVES, CATEGORIES, COMPOUNDING, CONS_MODES, CONSTRUCTION_HEADS,
    CONTINGENCY_BASES, COST_CURVES, CRORE, DEAL_TYPES, EDC_BASES, ENTITIES, HORIZON_Q, LAND_CURVES, LAUNCH_MODES,
    LINK_EVENTS, PROJECT_TYPES, SPARE_BASES, SQFT_PER_SQYD, SQYD_PER_ACRE, SQYD_PER_BIGHA, TRIGGERS, ZONES,
    CostHead, ClpPlan, Milestone, Product, Scenario, Timing, check_inputs, from_dict, sample_project, to_dict,
)
from engine.cashflow import run_model

st.set_page_config(page_title="Real Estate Feasibility Engine", page_icon="🏗️", layout="wide")

# ============================================================================ state & engine run
if "project" not in st.session_state:
    st.session_state.project = sample_project()
P = st.session_state.project


def rerun_after_edit(editor_key: str):
    """Commit pattern for editable tables: the project is already updated -> clear the table's edit
    memory and redraw everything from the project."""
    st.session_state.pop(editor_key, None)
    st.rerun()


FINGERPRINT = json.dumps(to_dict(P), sort_keys=True, default=str)   # project state used for this run's results
issues = check_inputs(P)
try:
    RES = run_model(P)
except Exception as exc:                      # an unforeseen input combination – never crash the screens
    RES, ENGINE_ERROR = None, str(exc)
else:
    ENGINE_ERROR = ""

PAGE_SECTIONS = {
    "1 · Project & land": ["Project", "Land"],
    "2 · Scenarios & zones": ["Scenarios", "Zones"],
    "3 · Products": ["Products"],
    "4 · Product timing": ["Timing"],
    "5 · Cost heads": ["Costs"],
    "6 · Calculators": ["BAR", "AIF", "Initial investment"],
    "7 · Payment plans": ["CLP"],
    "8 · Tax & benchmarks": ["Finance"],
    "9 · Checks & summary": [],
}


def issues_for(section: str, item: str | None = None) -> list:
    return [i for i in issues if i.section == section and (item is None or i.item == item)]


def show(section: str, item: str):
    """Show the error(s) for one field directly under it."""
    for i in issues_for(section, item):
        st.error(i.message, icon="⚠️")


def row_status(section: str, item: str) -> str:
    msgs = [i.message.split(": ", 1)[-1] for i in issues_for(section, item)]
    return "✔" if not msgs else "⚠ " + " | ".join(msgs)


def pct_in(label, value, key=None, help=None, max_value=100.0, step=0.25):
    """Percentage field: shown as 32 (%) but stored as 0.32."""
    return st.number_input(label, value=round(float(value) * 100, 6), min_value=-100.0, max_value=max_value,
                           step=step, format="%.2f", key=key, help=help) / 100


# ============================================================================ sidebar
with st.sidebar:
    st.title("🏗️ Feasibility Engine")
    st.caption(f"{P.info.name} · {P.info.project_type} · {P.info.deal_type} · {P.info.scenario}")
    labels = []
    for page, secs in PAGE_SECTIONS.items():
        n = sum(1 for i in issues if i.section in secs)
        labels.append(f"{page}  ⚠️ {n}" if n else page)
    choice = st.radio("Go to", labels, label_visibility="collapsed")
    page = choice.split("  ⚠️")[0]

    st.divider()
    engine_problems = RES["problems"] if RES else []
    if not issues and RES and not engine_problems:
        st.success("✔ ALL CHECKS PASSED")
    else:
        n_problems = len(issues) + sum(1 for p in engine_problems if not any(p.endswith(i.message) for i in issues))
        st.error(f"✖ {n_problems or 1} issue(s) – see page 9")
    if RES:
        m, pnl = RES["cashflow"].metrics, RES["cashflow"].pnl
        c1, c2 = st.columns(2)
        c1.metric("GDV", f"₹{pnl['revenue']:,.0f} Cr")
        c2.metric("EBITDA", f"{pnl['ebitda_margin']:.1%}")
        c1.metric("Developer IRR", "n/a" if m["developer_irr"] is None else f"{m['developer_irr']:.1%}")
        c2.metric("Peak funding", f"₹{m['peak_investment']:,.1f} Cr")
        st.caption(f"Payback: {'Q' + str(m['payback_q']) if m['payback_q'] else 'never'} · "
                   f"PAT ₹{pnl['pat']:,.1f} Cr · {RES['timeline'].last_q} quarters")

    st.divider()
    st.download_button("💾 Save project file", json.dumps(to_dict(P), indent=2, ensure_ascii=False),
                       file_name=f"{P.info.name.split(' (')[0].replace(' ', '_')}.json", mime="application/json",
                       width="stretch")
    up = st.file_uploader("📂 Open project file", type="json")
    if up is not None and st.session_state.get("loaded_file") != up.file_id:
        try:
            st.session_state.project = from_dict(json.load(up))
            st.session_state.loaded_file = up.file_id
            st.rerun()
        except Exception as exc:
            st.error(f"Could not open this file: {exc}")
    if st.button("↺ Reset to sample project", width="stretch"):
        st.session_state.project = sample_project()
        st.rerun()

# ============================================================================ page 1: project & land
if page.startswith("1"):
    st.header("1 · Project & land")
    st.caption("Fields with ⚠️ underneath need fixing. All amounts in ₹; land in sq yd.")
    a, b = st.columns(2)
    with a:
        st.subheader("Project")
        P.info.name = st.text_input("Project name", P.info.name)
        P.info.location = st.text_input("Location", P.info.location)
        P.info.project_type = st.selectbox("Project type", PROJECT_TYPES, PROJECT_TYPES.index(P.info.project_type),
                                           help="Only products whose zone belongs to this type are calculated (Mixed = all).")
        P.info.start_date = st.date_input("Project start = Q1 (quarter-end date)", P.info.start_date,
                                          min_value=date(2020, 1, 1), max_value=date(2045, 12, 31), format="DD/MM/YYYY")
        show("Project", "start_date")
        P.info.deal_type = st.radio("Deal type", DEAL_TYPES, DEAL_TYPES.index(P.info.deal_type), horizontal=True)
        P.info.entity = st.radio("Taxable entity", ENTITIES, ENTITIES.index(P.info.entity), horizontal=True)
        names = [s.name for s in P.scenarios]
        P.info.scenario = st.selectbox("Active scenario", names, names.index(P.info.scenario) if P.info.scenario in names else 0)
        show("Project", "scenario")
        ov = st.number_input("Lifecycle for ROCE / CAGR (years) – 0 = use project span", value=float(P.info.lifecycle_override_years or 0),
                             min_value=0.0, step=0.5)
        P.info.lifecycle_override_years = ov or None
    with b:
        st.subheader("Land")
        L = P.land
        L.area_sqyd = st.number_input("Land area (sq yd)", value=float(L.area_sqyd), min_value=0.0, step=100.0)
        st.caption(f"= {L.area_sqyd * SQFT_PER_SQYD:,.0f} sq ft · {L.area_sqyd / SQYD_PER_ACRE:,.2f} acres · "
                   f"{L.area_sqyd / SQYD_PER_BIGHA:,.2f} bigha")
        L.market_rate = st.number_input("Current market rate (₹ / sq yd)", value=float(L.market_rate), min_value=0.0, step=500.0)
        L.dlc_rate_sqm = st.number_input("DLC / circle rate (₹ / sq m)", value=float(L.dlc_rate_sqm), min_value=0.0, step=100.0)
        show("Land", "land_rates")
        if P.info.deal_type == "JV":
            L.lo_share = pct_in("Landowner revenue share (%)", L.lo_share)
            show("Land", "lo_share")
            L.stamp_duty_jv = pct_in("Stamp duty + registration on JV (% of DLC value)", L.stamp_duty_jv, step=0.05)
            if RES:
                cr = RES["costs"]
                lo_px = cr.land_cost * CRORE / L.area_sqyd if L.area_sqyd else 0
                st.info(f"Landowner effectively gets ₹{lo_px:,.0f} / sq yd = "
                        f"{lo_px / L.market_rate - 1:+.1%} vs today's market rate.", icon="💡")
        else:
            L.purchase_rate = st.number_input("Purchase rate (₹ / sq yd)", value=float(L.purchase_rate), min_value=0.0, step=500.0)
            show("Land", "purchase_rate")
            L.stamp_duty_outright = pct_in("Stamp duty + registration – outright (%)", L.stamp_duty_outright, step=0.05)
            c1, c2, c3 = st.columns(3)
            L.payment_curve = c1.selectbox("Land payment curve", LAND_CURVES, LAND_CURVES.index(L.payment_curve))
            L.payment_start_q = int(c2.number_input("Start quarter", value=int(L.payment_start_q), min_value=1, max_value=HORIZON_Q))
            L.payment_end_q = int(c3.number_input("End quarter", value=int(L.payment_end_q), min_value=1, max_value=HORIZON_Q,
                                                  disabled=L.payment_curve == "Upfront"))
            show("Land", "payment")

# ============================================================================ page 2: scenarios & zones
elif page.startswith("2"):
    st.header("2 · Scenarios & development zones")
    st.subheader("Scenarios")
    st.caption("Sales period multiplier 1.5 = products take 50% longer to sell. Delay pushes every fixed launch / construction start later.")
    df = pd.DataFrame([{"Scenario": s.name, "Sales period ×": s.sales_period_mult, "Price change %": s.price_change * 100,
                        "Construction cost change %": s.cost_change * 100, "Delay (quarters)": s.delay_q,
                        "Check": row_status("Scenarios", s.name)} for s in P.scenarios])
    ed = st.data_editor(df, key="ed_scen", hide_index=True, width="stretch", num_rows="dynamic",
                        column_config={"Sales period ×": st.column_config.NumberColumn(min_value=0.1, step=0.05, format="%.2f"),
                                       "Delay (quarters)": st.column_config.NumberColumn(min_value=0, step=1),
                                       "Check": st.column_config.TextColumn(disabled=True)})
    if not ed.equals(df):
        P.scenarios = [Scenario(str(r["Scenario"] or f"Scenario {i+1}"), float(r["Sales period ×"] or 1), float(r["Price change %"] or 0) / 100,
                                float(r["Construction cost change %"] or 0) / 100, int(r["Delay (quarters)"] or 0))
                       for i, r in ed.iterrows()]
        rerun_after_edit("ed_scen")
    show("Scenarios", "names")

    st.subheader("Development zones")
    used = {"Villa": ["Villa Development"], "Apartments": ["Apartment Development"], "Plotted": ["Plotted Development"]}.get(P.info.project_type, ZONES)
    df = pd.DataFrame([{"Zone": z.name, "Share of land %": z.share * 100,
                        "Land area (sq ft)": round(z.share * P.land.area_sqyd * SQFT_PER_SQYD),
                        "In this project type?": "Yes" if (z.name in used or z.name == "Other / Clubhouse") else "No"} for z in P.zones])
    ed = st.data_editor(df, key="ed_zone", hide_index=True, width="stretch", num_rows="fixed",
                        column_config={"Zone": st.column_config.TextColumn(disabled=True),
                                       "Land area (sq ft)": st.column_config.NumberColumn(disabled=True, format="%d"),
                                       "In this project type?": st.column_config.TextColumn(disabled=True)})
    if not ed.equals(df):
        for z, (_, r) in zip(P.zones, ed.iterrows()):
            z.share = float(r["Share of land %"] or 0) / 100
        rerun_after_edit("ed_zone")
    st.caption(f"Total share: {sum(z.share for z in P.zones):.1%}")
    show("Zones", "zones")

# ============================================================================ page 3: products
elif page.startswith("3"):
    st.header("3 · Product mix")
    st.caption("One row per product. Add a row with the ➕ at the bottom of the table; tick **Include** to use it. "
               "**Active** is automatic: included AND its zone fits the project type.")
    active = {p.name for p in P.active_products()}
    plans = [pl.name for pl in P.plans]
    rev = RES["curves"].revenue if RES else {}
    df = pd.DataFrame([{
        "Include": p.include, "Product": p.name, "Zone": p.zone, "Category": p.category,
        "Area (sq ft)": p.area_sqft, "Sales rate (₹/sq ft)": p.rate, "Construction (₹/sq ft)": p.construction_rate,
        "Launch Q (fixed)": p.launch_q, "Sales period (qtrs)": p.sales_period_q, "Sales curve": p.sales_curve,
        "Payment plan": p.clp_plan, "Construction start Q (fixed)": p.construction_start_q,
        "Construction (qtrs)": p.construction_duration_q, "Construction curve": p.construction_curve,
        "Active": "Yes" if p.name in active else ("Excluded by type" if p.include else "Off"),
        "Revenue (₹ Cr)": round(rev.get(p.name, 0.0), 2), "Check": row_status("Products", p.name)} for p in P.products])
    num = lambda **k: st.column_config.NumberColumn(**k)
    ed = st.data_editor(df, key="ed_prod", hide_index=True, width="stretch", num_rows="dynamic", column_config={
        "Include": st.column_config.CheckboxColumn(default=True),
        "Zone": st.column_config.SelectboxColumn(options=ZONES, required=True, default="Apartment Development"),
        "Category": st.column_config.SelectboxColumn(options=CATEGORIES, required=True, default="Apartment"),
        "Area (sq ft)": num(min_value=0, step=1000, format="%d"), "Sales rate (₹/sq ft)": num(min_value=0, step=100, format="%d"),
        "Construction (₹/sq ft)": num(min_value=0, step=50, format="%d"),
        "Launch Q (fixed)": num(min_value=1, max_value=HORIZON_Q, step=1), "Sales period (qtrs)": num(min_value=1, max_value=HORIZON_Q, step=1),
        "Sales curve": st.column_config.SelectboxColumn(options=BASIC_CURVES, required=True, default="Even"),
        "Payment plan": st.column_config.SelectboxColumn(options=plans, required=True, default=plans[0] if plans else None),
        "Construction start Q (fixed)": num(min_value=1, max_value=HORIZON_Q, step=1),
        "Construction (qtrs)": num(min_value=1, max_value=HORIZON_Q, step=1),
        "Construction curve": st.column_config.SelectboxColumn(options=BASIC_CURVES, required=True, default="S-curve"),
        "Active": st.column_config.TextColumn(disabled=True), "Revenue (₹ Cr)": num(disabled=True, format="%.2f"),
        "Check": st.column_config.TextColumn(disabled=True, width="large")})
    if not ed.equals(df):
        old = {i: p for i, p in enumerate(P.products)}
        new = []
        for i, r in ed.reset_index(drop=True).iterrows():
            v = lambda col, d: d if pd.isna(r[col]) else r[col]
            prev = old.get(i)
            name = str(v("Product", f"New product {i+1}"))
            prod = Product(name, v("Zone", "Apartment Development"), v("Category", "Apartment"), bool(v("Include", True)),
                           float(v("Area (sq ft)", 0)), float(v("Sales rate (₹/sq ft)", 0)), float(v("Construction (₹/sq ft)", 0)),
                           int(v("Launch Q (fixed)", 1)), int(v("Sales period (qtrs)", 1)), v("Sales curve", "Even"),
                           v("Payment plan", plans[0] if plans else ""), int(v("Construction start Q (fixed)", 1)),
                           int(v("Construction (qtrs)", 1)), v("Construction curve", "S-curve"),
                           copy.deepcopy(prev.timing) if prev else Timing())
            if prev and prev.name != name:                       # renamed -> keep links pointing to it
                for q in P.products:
                    for attr in ("launch_link_product", "cons_link_product"):
                        if getattr(q.timing, attr) == prev.name: setattr(q.timing, attr, name)
                for q in new:
                    for attr in ("launch_link_product", "cons_link_product"):
                        if getattr(q.timing, attr) == prev.name: setattr(q.timing, attr, name)
            new.append(prod)
        P.products = new
        rerun_after_edit("ed_prod")
    show("Products", "names"); show("Products", "active")
    if RES:
        st.caption(f"Active saleable area {RES['costs'].total_sa:,.0f} sq ft · GDV ₹{RES['costs'].total_revenue:,.2f} Cr · "
                   f"construction ₹{RES['costs'].total_construction:,.2f} Cr")

# ============================================================================ page 4: product timing
elif page.startswith("4"):
    st.header("4 · Product timing – how each launch and construction start is decided")
    st.caption("Fixed = uses the quarter typed on page 3. Linked = X quarters after an event of a product listed ABOVE it "
               "(put the lead product first). Offset 0 = same quarter, 1 = next quarter, −1 = one before.")
    names = [p.name for p in P.products]
    notes = {n: t.note for n, t in RES["timeline"].products.items()} if RES else {}
    df = pd.DataFrame([{
        "Product": p.name, "Launch timing": p.timing.launch_mode, "Launch – linked product": p.timing.launch_link_product,
        "Launch – event": p.timing.launch_link_event, "Launch offset": p.timing.launch_offset,
        "Construction timing": p.timing.cons_mode, "Construction – linked product": p.timing.cons_link_product,
        "Construction – event": p.timing.cons_link_event, "Construction offset": p.timing.cons_offset,
        "Result": notes.get(p.name, "— (not active)"), "Check": row_status("Timing", p.name)} for p in P.products])
    sel = lambda opts: st.column_config.SelectboxColumn(options=opts)
    ed = st.data_editor(df, key="ed_time", hide_index=True, width="stretch", num_rows="fixed", column_config={
        "Product": st.column_config.TextColumn(disabled=True), "Launch timing": sel(LAUNCH_MODES),
        "Launch – linked product": sel(names), "Launch – event": sel(LINK_EVENTS),
        "Launch offset": st.column_config.NumberColumn(min_value=-8, max_value=40, step=1),
        "Construction timing": sel(CONS_MODES), "Construction – linked product": sel(names), "Construction – event": sel(LINK_EVENTS),
        "Construction offset": st.column_config.NumberColumn(min_value=-8, max_value=40, step=1),
        "Result": st.column_config.TextColumn(disabled=True, width="large"), "Check": st.column_config.TextColumn(disabled=True)})
    if not ed.equals(df):
        for p, (_, r) in zip(P.products, ed.iterrows()):
            n = lambda col, d=None: d if pd.isna(r[col]) else r[col]
            p.timing = Timing(n("Launch timing", "Fixed quarter"), n("Launch – linked product"), n("Launch – event"), int(n("Launch offset", 0)),
                              n("Construction timing", "Fixed quarter"), n("Construction – linked product"), n("Construction – event"),
                              int(n("Construction offset", 0)))
        rerun_after_edit("ed_time")
    if RES:
        for e in RES["timeline"].errors: st.error(e, icon="⚠️")
        tl = RES["timeline"]
        st.subheader("Timeline (S = selling · C = construction · # = both)")
        rows = []
        for t in tl.products.values():
            rows.append({"Product": t.name, **{f"Q{q}": ("#" if (t.launch <= q <= t.sales_end and t.cons_start <= q <= t.cons_end)
                                                      else "S" if t.launch <= q <= t.sales_end else "C" if t.cons_start <= q <= t.cons_end else "")
                                               for q in range(1, max(tl.last_q, 1) + 1)}})
        st.dataframe(pd.DataFrame(rows), hide_index=True, width="stretch")

# ============================================================================ page 5: cost heads
elif page.startswith("5"):
    st.header("5 · Cost heads")
    st.caption("Change rates, switch heads on/off and pick how each cost is spread over time. Grey 'Auto' rows are calculated "
               "elsewhere (land page, product table, calculators). Percent inputs are typed as % (4 = 4%).")
    META = {  # key: (how it is calculated, unit of input 1, unit of input 2)
        "land": ("JV: landowner share × revenue | Outright: price × area", "Auto", ""), "sd": ("From land page", "Auto", ""),
        "brok": ("% × land value", "%", ""), "ovh": ("% × total revenue", "%", ""),
        "mkt": ("Input 1 % × built-up revenue + Input 2 % × plotted revenue", "%", "%"),
        "legal": ("₹/sq ft × total saleable area", "₹/sq ft", ""), "ews": ("Loss ₹/sq ft × EWS share × saleable area", "₹/sq ft", "%"),
        "cont": ("% × base chosen in 'Basis'", "%", ""), "edc": ("Input 1 per 'Basis'", "per basis", ""),
        "hotel": ("Fee per key (₹) × number of keys", "₹/key", "keys"), "furn": ("₹/sq ft × studio area", "₹/sq ft", ""),
        "gst": ("(Commercial + plot revenue) × unsold % × GST % × developer share", "% GST", "% unsold"),
        "aif": ("Calculator page 6", "Auto", ""), "bar": ("Calculator page 6", "Auto", "")}
    for k, n, cat in CONSTRUCTION_HEADS: META[k] = (f"Σ {cat} products (page 3)", "Auto", "")
    PCT1 = {"brok", "ovh", "mkt", "cont", "gst"}; PCT2 = {"mkt", "ews", "gst"}
    amounts = RES["costs"].amounts if RES else {}
    tsa = RES["costs"].total_sa if RES else 0
    def shown1(c):
        if c.rate1 is None: return None
        is_pct = c.key in PCT1 or (c.key == "edc" and c.rate2 == "% of construction cost") or \
                 (c.key.startswith("spare") and c.basis in ("% of total revenue", "% of construction cost"))
        return c.rate1 * 100 if is_pct else c.rate1
    df = pd.DataFrame([{
        "Cost head": c.name, "How it is calculated": META.get(c.key, ("Spare: per 'Basis'", "per basis", ""))[0],
        "Input 1": shown1(c), "Unit 1": META.get(c.key, ("", "per basis", ""))[1],
        "Input 2": (c.rate2 * 100 if c.key in PCT2 else c.rate2) if isinstance(c.rate2, (int, float)) else None,
        "Unit 2": META.get(c.key, ("", "", ""))[2],
        "Basis": c.rate2 if c.key in ("cont", "edc") else (c.basis if c.key.startswith("spare") else None),
        "Applicable": c.applicable,
        "Phasing curve": ("Follow Collections" if P.info.deal_type == "JV" else P.land.payment_curve) if c.key == "land" else c.curve,
        "Start Q": c.start_q, "End Q": c.end_q,
        "Amount (₹ Cr)": round(amounts.get(c.key, 0.0), 2),
        "₹ / sq ft": round(amounts.get(c.key, 0.0) * CRORE / tsa, 0) if tsa else 0.0,
        "Check": row_status("Costs", c.key)} for c in P.costs])
    ed = st.data_editor(df, key="ed_cost", hide_index=True, width="stretch", num_rows="fixed", column_config={
        "Cost head": st.column_config.TextColumn(width="medium"), "How it is calculated": st.column_config.TextColumn(disabled=True),
        "Unit 1": st.column_config.TextColumn(disabled=True), "Unit 2": st.column_config.TextColumn(disabled=True),
        "Basis": st.column_config.SelectboxColumn(options=sorted(set(SPARE_BASES + EDC_BASES + CONTINGENCY_BASES))),
        "Phasing curve": st.column_config.SelectboxColumn(options=COST_CURVES),
        "Start Q": st.column_config.NumberColumn(min_value=1, max_value=HORIZON_Q, step=1),
        "End Q": st.column_config.NumberColumn(min_value=1, max_value=HORIZON_Q, step=1),
        "Amount (₹ Cr)": st.column_config.NumberColumn(disabled=True, format="%.2f"),
        "₹ / sq ft": st.column_config.NumberColumn(disabled=True, format="%d"),
        "Check": st.column_config.TextColumn(disabled=True, width="large")})
    if not ed.equals(df):
        auto = {"land", "sd", "aif", "bar"} | {k for k, _, _ in CONSTRUCTION_HEADS}
        for c, (_, r) in zip(P.costs, ed.iterrows()):
            g = lambda col: None if pd.isna(r[col]) else r[col]
            if c.key.startswith("spare"): c.name = str(g("Cost head") or c.name); c.basis = g("Basis")
            c.applicable = bool(r["Applicable"])
            if c.key != "land":
                c.curve = g("Phasing curve") or c.curve
                c.start_q = None if g("Start Q") is None else int(g("Start Q"))
                c.end_q = None if g("End Q") is None else int(g("End Q"))
            if c.key in auto: continue
            v1 = g("Input 1")
            if v1 is not None:
                is_pct = c.key in PCT1 or (c.key == "edc" and (g("Basis") or c.rate2) == "% of construction cost") or \
                         (c.key.startswith("spare") and c.basis in ("% of total revenue", "% of construction cost"))
                c.rate1 = float(v1) / 100 if is_pct else float(v1)
            if c.key in ("cont", "edc"):
                c.rate2 = g("Basis") or c.rate2
            elif c.key in PCT2 or c.key == "hotel":
                v2 = g("Input 2")
                c.rate2 = (float(v2) / 100 if c.key in PCT2 else float(v2)) if v2 is not None else c.rate2
        rerun_after_edit("ed_cost")
    if st.button("➕ Add a spare cost head"):
        n = sum(1 for c in P.costs if c.key.startswith("spare")) + 1
        P.costs.append(CostHead(f"spare{n}", f"Other cost {n}", rate1=0.0, applicable=True, curve="Follow Construction",
                                basis="Lump sum (Rs Cr)"))
        rerun_after_edit("ed_cost")
    if RES:
        st.metric("TOTAL COST", f"₹{RES['costs'].total_cost:,.2f} Cr", f"{RES['costs'].total_cost / RES['costs'].total_revenue:.1%} of GDV",
                  delta_color="off")

# ============================================================================ page 6: calculators
elif page.startswith("6"):
    st.header("6 · Calculators")
    t1, t2, t3 = st.tabs(["BAR / premium FAR", "AIF cost", "Initial investment"])
    with t1:
        b = P.bar
        c1, c2 = st.columns(2)
        b.authority = c1.selectbox("Authority", BAR_AUTHORITIES, BAR_AUTHORITIES.index(b.authority))
        b.land_use = c2.selectbox("Land use", BAR_LAND_USES, BAR_LAND_USES.index(b.land_use))
        b.free_far = c1.number_input("Free FAR (× land area)", value=float(b.free_far), min_value=0.0, step=0.1)
        b.loading_factor = c2.number_input("Loading factor (saleable ÷ built-up)", value=float(b.loading_factor), min_value=0.01, step=0.05)
        b.reserved_price_sqm = c1.number_input("Reserved price (₹ / sq m)", value=float(b.reserved_price_sqm), min_value=0.0, step=100.0)
        b.gst_applicable = c2.checkbox("GST on BAR applicable", b.gst_applicable)
        b.gst_rate = pct_in("GST rate on BAR (%)", b.gst_rate)
        st.write("**BAR reserved-price % table** (blank = combination not allowed)")
        df = pd.DataFrame({a: {u: (None if b.pct_table[u].get(a) is None else b.pct_table[u][a] * 100) for u in BAR_LAND_USES}
                           for a in BAR_AUTHORITIES})
        ed = st.data_editor(df, key="ed_bar")
        if not ed.equals(df):
            b.pct_table = {u: {a: (None if pd.isna(ed.loc[u, a]) else float(ed.loc[u, a]) / 100) for a in BAR_AUTHORITIES} for u in BAR_LAND_USES}
            rerun_after_edit("ed_bar")
        show("BAR", "bar")
        if RES:
            x = RES["costs"].bar
            st.success(f"Free {x['free_sqm']:,.0f} sq m · planned {x['planned_sqm']:,.0f} sq m · extra {x['extra_sqm']:,.0f} sq m × "
                       f"₹{b.reserved_price_sqm:,.0f} × {x['pct']:.0%} = **₹{x['cost']:.2f} Cr**")
    with t2:
        a = P.aif
        a.rate = pct_in("Interest rate (% p.a.)", a.rate)
        a.tenure_years = st.number_input("Tenure (years)", value=float(a.tenure_years), min_value=0.0, step=0.5)
        a.compounding = st.selectbox("Compounding", list(COMPOUNDING), list(COMPOUNDING).index(a.compounding))
        show("AIF", "aif")
        if RES:
            x = RES["costs"].aif
            st.info(f"Principal (land + stamp duty + brokerage) ₹{x['principal']:,.2f} Cr → interest **₹{x['cost']:,.2f} Cr**. "
                    f"Switch the 'AIF Cost' head on (page 5) to include it.", icon="💡")
    with t3:
        ii = P.initial
        st.write("**JV – advance to landowner and early payments** (only used in JV deals)")
        c1, c2, c3 = st.columns(3)
        ii.advance_pct = pct_in("Advance to landowner (% of land value at market)", ii.advance_pct)
        ii.dd_amount = c1.number_input("Paid on due diligence (₹ Cr)", value=float(ii.dd_amount), min_value=0.0, step=0.5)
        ii.da_amount = c2.number_input("Paid at DA (₹ Cr)", value=float(ii.da_amount), min_value=0.0, step=0.5)
        show("Initial investment", "dd_da")
        ii.legal_share = pct_in("Share of legal & compliance paid before DA (%)", ii.legal_share)
        ii.bar_share = pct_in("Share of BAR paid upfront (%)", ii.bar_share)
        ii.dev_cost_pre_da = c3.number_input("Development cost before DA (₹ Cr)", value=float(ii.dev_cost_pre_da), min_value=0.0, step=0.5)
        ii.marketing_pre_launch = c3.number_input("Pre-launch marketing (₹ Cr)", value=float(ii.marketing_pre_launch), min_value=0.0, step=0.5)
        labels = {"token": "Term-sheet token (balance)", "dd": "Due diligence", "da": "At DA", "legal": "Legal & compliance",
                  "dev": "Pre-DA development", "marketing": "Pre-launch marketing", "sd": "Stamp duty", "brok": "Brokerage", "bar": "BAR (upfront share)"}
        items = RES["costs"].initial_items if RES else {}
        df = pd.DataFrame([{"Item": labels[k], "Amount (₹ Cr)": round(items.get(k, 0.0), 2), "Payout quarter": q}
                           for k, q in ii.payout_q.items()])
        ed = st.data_editor(df, key="ed_ii", hide_index=True, num_rows="fixed", column_config={
            "Item": st.column_config.TextColumn(disabled=True), "Amount (₹ Cr)": st.column_config.NumberColumn(disabled=True, format="%.2f"),
            "Payout quarter": st.column_config.NumberColumn(min_value=1, max_value=HORIZON_Q, step=1)})
        if not ed.equals(df):
            for k, (_, r) in zip(list(ii.payout_q), ed.iterrows()):
                ii.payout_q[k] = int(r["Payout quarter"]) if not pd.isna(r["Payout quarter"]) else ii.payout_q[k]
            rerun_after_edit("ed_ii")
        show("Initial investment", "payout_q")
        st.write("**Outright – initial outlay** = land + stamp duty + brokerage + shares below")
        ii.outright_construction_share = pct_in("Share of built-up construction (%)", ii.outright_construction_share)
        ii.outright_legal_share = pct_in("Share of legal & compliance (%)", ii.outright_legal_share)
        if RES:
            cr = RES["costs"]
            st.success(f"Initial investment – JV ₹{cr.initial_jv:,.2f} Cr · Outright ₹{cr.initial_outright:,.2f} Cr · "
                       f"**used ({P.info.deal_type}) ₹{cr.initial_used:,.2f} Cr**")

# ============================================================================ page 7: payment plans
elif page.startswith("7"):
    st.header("7 · Payment plans (CLP)")
    st.caption("Booking-linked: value = quarters after booking (0–12). Construction-linked: value = % of that product's construction "
               "(e.g. 35). Each plan must total exactly 100%. Add as many milestones as you need with ➕.")
    P.catch_up_lag = int(st.number_input("Catch-up lag – late buyers pay milestones already reached this many quarters after booking",
                                         value=int(P.catch_up_lag), min_value=0, max_value=4, step=1))
    show("CLP", "catch_up_lag")
    names = [pl.name for pl in P.plans]
    used_by = {pl: [p.name for p in P.products if p.clp_plan == pl] for pl in names}
    c1, c2 = st.columns([2, 1])
    choice = c1.selectbox("Plan", names, format_func=lambda n: f"{n}  {'⚠️' if issues_for('CLP', n) else '✔'}  "
                                                              f"(used by: {', '.join(used_by[n]) or 'none'})")
    plan = P.plan(choice)
    new_name = c2.text_input("Rename plan", plan.name)
    if new_name and new_name != plan.name and new_name not in names:
        for p in P.products:
            if p.clp_plan == plan.name: p.clp_plan = new_name
        plan.name = new_name
        st.rerun()
    df = pd.DataFrame([{"Milestone": m.name, "Trigger": m.trigger,
                        "Value": m.value if m.trigger == "Booking + quarters" else round(m.value * 100, 4),
                        "% of sale": round(m.pct * 100, 4)} for m in plan.milestones])
    if df.empty:
        df = pd.DataFrame(columns=["Milestone", "Trigger", "Value", "% of sale"])
    ed = st.data_editor(df, key=f"ed_plan_{choice}", hide_index=True, width="stretch", num_rows="dynamic", column_config={
        "Trigger": st.column_config.SelectboxColumn(options=TRIGGERS, required=True, default="Construction % complete"),
        "Value": st.column_config.NumberColumn(min_value=0, max_value=100, help="quarters (booking) or % (construction)"),
        "% of sale": st.column_config.NumberColumn(min_value=0, max_value=100, step=0.5, format="%.2f")})
    if not ed.equals(df):
        ms = []
        for _, r in ed.iterrows():
            if pd.isna(r["Milestone"]) and pd.isna(r["% of sale"]): continue
            trig = r["Trigger"] if not pd.isna(r["Trigger"]) else "Construction % complete"
            val = 0.0 if pd.isna(r["Value"]) else float(r["Value"])
            ms.append(Milestone(str(r["Milestone"]) if not pd.isna(r["Milestone"]) else "Milestone", trig,
                                val if trig == "Booking + quarters" else val / 100, 0.0 if pd.isna(r["% of sale"]) else float(r["% of sale"]) / 100))
        plan.milestones = ms
        rerun_after_edit(f"ed_plan_{choice}")
    total = sum(m.pct for m in plan.milestones)
    (st.success if abs(total - 1) < 1e-6 else st.error)(f"Plan total: {total:.2%}" + ("" if abs(total - 1) < 1e-6 else " – must be exactly 100%"))
    for i in issues_for("CLP", plan.name):
        if "totals" not in i.message: st.error(i.message, icon="⚠️")
    if RES:
        sched = [s for s in RES["collections"].schedule if s.plan == plan.name and s.due_q]
        if sched:
            st.write("**When construction-linked milestones fall due** (for products using this plan)")
            st.dataframe(pd.DataFrame([{"Product": s.product, "Milestone": s.name, "Falls due": s.text} for s in sched]),
                         hide_index=True, width="stretch")
    st.divider()
    c1, c2, c3 = st.columns(3)
    nn = c1.text_input("New plan name", placeholder="e.g. CLP – 12-Floor Tower")
    if c1.button("➕ Create empty plan") and nn and nn not in names:
        P.plans.append(ClpPlan(nn, [])); st.rerun()
    if c2.button("⧉ Duplicate this plan"):
        k = 2
        while f"{plan.name} ({k})" in names: k += 1
        P.plans.append(ClpPlan(f"{plan.name} ({k})", copy.deepcopy(plan.milestones))); st.rerun()
    if c3.button("🗑 Delete this plan", disabled=bool(used_by[plan.name]),
                 help="Only plans not used by any product can be deleted."):
        P.plans = [pl for pl in P.plans if pl.name != plan.name]; st.rerun()
    show("CLP", "plan_names")

# ============================================================================ page 8: tax & benchmarks
elif page.startswith("8"):
    st.header("8 · Tax, cost of equity & KPI benchmarks")
    f = P.finance
    c1, c2, c3 = st.columns(3)
    with c1:
        f.tax_company = pct_in("Tax rate – Company (%)", f.tax_company, help="Verify with Finance (statutory 115BAA ≈ 25.17%)")
        f.tax_llp = pct_in("Tax rate – LLP (%)", f.tax_llp, help="Verify with Finance (statutory ≈ 34.94%)")
        f.cost_of_equity = pct_in("Cost of equity / discount rate (%)", f.cost_of_equity)
        show("Finance", "finance")
    labels = {"project_irr": "Project IRR (%)", "developer_irr": "Developer IRR (%)", "roce": "ROCE (%)", "cagr": "CAGR (%)",
              "pat_margin": "PAT margin (%)", "ebitda_margin": "EBITDA margin (%)"}
    with c2:
        st.write("**Benchmarks** (green when beaten)")
        for k in ["project_irr", "developer_irr", "roce"]: f.benchmarks[k] = pct_in(labels[k], f.benchmarks[k], key=f"bm_{k}")
    with c3:
        st.write("&nbsp;")
        for k in ["cagr", "pat_margin", "ebitda_margin"]: f.benchmarks[k] = pct_in(labels[k], f.benchmarks[k], key=f"bm_{k}")
        f.benchmarks["equity_multiple"] = st.number_input("Equity multiple (x)", value=float(f.benchmarks["equity_multiple"]), min_value=0.0, step=0.1)

# ============================================================================ page 9: checks & summary
else:
    st.header("9 · Checks & summary")
    if not issues and RES and not RES["problems"]:
        st.success("✔ ALL CHECKS PASSED – the results below can be used.")
    else:
        for page_name, secs in PAGE_SECTIONS.items():
            mine = [i for i in issues if i.section in secs]
            if mine:
                st.subheader(page_name)
                for i in mine: st.error(i.message, icon="⚠️")
        if RES:
            for p in RES["problems"]:
                if not any(p.endswith(i.message) for i in issues): st.error(p, icon="⚠️")
        if ENGINE_ERROR:
            st.error(f"The engine could not run with these inputs: {ENGINE_ERROR}")
        if RES is None:
            st.info("Results appear once the product, timing and payment-plan errors above are fixed.")
    if RES:
        pnl, m = RES["cashflow"].pnl, RES["cashflow"].metrics
        st.subheader("Headline results")
        c = st.columns(4)
        c[0].metric("GDV", f"₹{pnl['revenue']:,.1f} Cr"); c[1].metric("Total cost", f"₹{pnl['total_cost']:,.1f} Cr")
        c[2].metric("EBITDA", f"₹{pnl['ebitda']:,.1f} Cr", f"{pnl['ebitda_margin']:.1%} margin", delta_color="off")
        c[3].metric("PAT", f"₹{pnl['pat']:,.1f} Cr", f"{pnl['pat_margin']:.1%} margin", delta_color="off")
        c = st.columns(4)
        fmt = lambda v: "n/a" if v is None else f"{v:.1%}"
        c[0].metric("Developer IRR", fmt(m["developer_irr"])); c[1].metric("Developer NPV", f"₹{m['developer_npv']:,.1f} Cr")
        c[2].metric("Peak investment", f"₹{m['peak_investment']:,.1f} Cr", f"in Q{m['peak_quarter']}" if m["peak_quarter"] else "", delta_color="off")
        c[3].metric("Payback", f"Q{m['payback_q']}" if m["payback_q"] else "never",
                    m["payback_date"].strftime("%b %Y") if m["payback_date"] else "", delta_color="off")
        st.caption("The full dashboard with charts comes in Step 7.")

# ============================================================================ keep results in step with the inputs
# Fields above change the project while this page is drawn; if anything changed, redraw once so every
# number, check and error message reflects the new inputs.
if json.dumps(to_dict(P), sort_keys=True, default=str) != FINGERPRINT:
    st.rerun()


### Cell 6 – ▶ Start the app and get your link
This starts the app inside Colab and opens a secure temporary tunnel (Cloudflare 'quick tunnel', free, no account).
**Click the link that appears** – the app opens in a new tab. Keep this notebook open while you use the app.

*The link changes every time you run this cell and stops when Colab disconnects. A permanent link comes in Step 10.*

In [ ]:
import subprocess, time, re, os, urllib.request

subprocess.run('pkill -f "streamlit run" ; pkill -f cloudflared', shell=True)
time.sleep(1)
app = subprocess.Popen(['streamlit', 'run', 'app.py', '--server.port', '8501', '--server.headless', 'true'],
                       cwd=BASE, stdout=open('/content/streamlit.log', 'w'), stderr=subprocess.STDOUT)

CF = '/content/cloudflared'
if not os.path.exists(CF):
    urllib.request.urlretrieve('https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', CF)
    os.chmod(CF, 0o755)
tunnel = subprocess.Popen([CF, 'tunnel', '--url', 'http://localhost:8501', '--no-autoupdate'],
                          stdout=open('/content/tunnel.log', 'w'), stderr=subprocess.STDOUT)

url = None
for _ in range(60):                       # wait up to ~60 seconds for the link
    time.sleep(1)
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', open('/content/tunnel.log').read())
    if m: url = m.group(0); break
if url:
    print('✅ App is running. Open it here (it may take ~10 seconds to load the first time):\n')
    print('   ', url)
else:
    print('⚠️ Tunnel link not ready yet – re-run this cell, or use Cell 7 (Colab built-in link).')

### Cell 7 – Alternative link (only if Cell 6 did not work)
Uses Colab's own built-in proxy. It opens only in **your** browser while you are signed in to Google.

In [ ]:
import subprocess, time
subprocess.run('pkill -f "streamlit run"', shell=True); time.sleep(1)
subprocess.Popen(['streamlit', 'run', 'app.py', '--server.port', '8501', '--server.headless', 'true',
                  '--server.enableCORS', 'false', '--server.enableXsrfProtection', 'false'],
                 cwd=BASE, stdout=open('/content/streamlit.log', 'w'), stderr=subprocess.STDOUT)
time.sleep(5)
try:
    from google.colab.output import eval_js
    print('Open the app here:', eval_js('google.colab.kernel.proxyPort(8501)'))
except ImportError:
    print('Not running in Colab – open http://localhost:8501')

### Cell 8 – Try these in the app (2-minute tour)
1. **Sidebar** – note GDV ₹803 Cr, developer IRR 110.2%, peak funding ₹34.3 Cr, and ✔ ALL CHECKS PASSED.
2. **1 · Project & land** – switch *Deal type* to **Outright** → developer IRR drops to ~31%, peak funding jumps to ~₹208 Cr. Switch back to JV.
3. **1 · Project & land** – change the start date to **15 Jul 2027** → a ⚠️ appears right under the date and the sidebar shows *1 · Project & land ⚠️ 1*. Change it back to 30 Jun 2027.
4. **3 · Products** – tick **Include** for *Residential plots* → GDV rises to ₹870 Cr. Clear its *Area* → the **Check** column of that row explains the problem.
5. **7 · Payment plans** – open *CLP – Apartment*, change a % → the plan total turns red and the plan shows ⚠️ in the list. Fix it back to 100%.
6. **4 · Product timing** – read the *Result* column: every launch/construction quarter explained in plain English.
7. **💾 Save project file** in the sidebar, then **📂 Open project file** to load it back.

### Cell 9 – If something looks wrong: the last lines of the app log

In [ ]:
print(open('/content/streamlit.log').read()[-3000:])

### Cell 10 – Stop the app when you are done

In [ ]:
import subprocess
subprocess.run('pkill -f "streamlit run" ; pkill -f cloudflared', shell=True)
print('App stopped.')

---
### ✅ Step 6 complete
`feasibility_app/` now has `app.py` (the input screens) next to `engine/`, `tests/` and `data/`.

**Next – Step 7:** the dashboard screen – KPI cards with benchmark colours, auto-written takeaways and interactive charts
(cash flow & funding curve, cost breakdown, revenue mix, collections vs costs, Gantt).